In [ ]:
import os
import logging
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)
BASE_DIR = os.getcwd() 
DEFAULT_CACHE_PATH = os.path.join(BASE_DIR, 'data', 'beijing_aqi.csv')

In [ ]:
def get_standardized_data(cache_path=DEFAULT_CACHE_PATH, force_download=False):
    # 1. Check Cache
    if os.path.exists(cache_path) and not force_download:
        logger.info(f"Loading standardized data from cache: {cache_path}")
        return pd.read_csv(cache_path, index_col='datetime', parse_dates=True)

    # 2. Fetch Data
    logger.info("Fetching dataset from UCI Repository...")
    try:
        data = fetch_ucirepo(id=381)
        df = pd.concat([data.data.features, data.data.targets], axis=1)
    except Exception as e:
        logger.error(f"Critical Ingestion Error: {e}")
        return None

    # 3. Standardize Schema & Time
    # Ensure the datetime index is created exactly the same way every time
    df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
    df.set_index('datetime', inplace=True)
    df.drop(['year', 'month', 'day', 'hour'], axis=1, inplace=True)
    
    # 4. Handle Missing Values (Linear Interpolation for Time Series)
    null_count = df['pm2.5'].isnull().sum()
    if null_count > 0:
        logger.info(f"Interpolating {null_count} missing PM2.5 values.")
        df['pm2.5'] = df['pm2.5'].interpolate(method='linear')
        df = df.dropna(subset=['pm2.5'])

    # 5. Quality Check
    if null_count > (0.10 * len(df)): # Warning if >10% was missing
        logger.warning("Data quality alert: Significant portion of data was missing/interpolated.")

    # 6. Save Standardized Output
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    df.to_csv(cache_path)
    logger.info(f"Standardized data saved to {cache_path}")

    return df

In [ ]:
df = get_standardized_data(force_download=True) # Test a fresh download

if df is not None:
    print("--- SUCCESS: PIPELINE TEST ---")
    print(f"Index Type: {df.index.dtype}") # Should be datetime64[ns]
    print(f"Missing PM2.5: {df['pm2.5'].isna().sum()}") # Should be 0
    print(f"Shape: {df.shape}")
    display(df.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
df['pm2.5'].iloc[:500].plot(title="Standardized PM2.5 (First 500 hours)")
plt.ylabel("PM2.5 Concentration")
plt.show()